### 1. Campaign Population
- Reporting year: 2025
- BusinessRule: campaign must start AND end within 2025
- Expected: 64 campaigns

In [0]:
%sql
SELECT COUNT(*) AS total_campaings, MIN(start_date) AS earlist_start_date, MAX(end_date) AS latest_end_date FROM `campaign&promotion`.gold.dim_campaign WHERE start_date >= DATE '2025-01-01' AND end_date <= DATE '2025-12-31';

### 2. Campaign Type Distribution

**Business View => What kind of campaigns did marketing actually run in 2025?**

In [0]:
%sql
SELECT campaign_type, COUNT(*) AS campaign_count, SUM(budget) AS total_budget,
ROUND(AVG(budget),0) AS avg_budget
FROM `campaign&promotion`.gold.dim_campaign WHERE start_date >= DATE '2025-01-01' AND end_date <= DATE '2025-12-31'
GROUP BY campaign_type
ORDER BY campaign_count DESC;


### 3. Check campaign data validity

Expected : invalid_campaign_dates = 0


In [0]:
%sql
SELECT
    COUNT(*) AS invalid_campaign_dates
FROM 
`campaign&promotion`.gold.dim_campaign
WHERE end_date < start_date;

### 4. Check duplicate campaign IDs

Expected : zero rows

In [0]:
%sql
SELECT
    campaign_id,
    COUNT(*) AS row_count
FROM `campaign&promotion`.gold.dim_campaign
GROUP BY campaign_id
HAVING row_count > 1;

### 5.Customer Profile

In [0]:
%sql
SELECT 
    COUNT(*) AS customer_count,
    COUNT(DISTINCT customer_id) AS distinct_customers,
    MIN(registration_date) AS earliest_transaction,
    MAX(registration_date) AS latest_transaction
FROM `campaign&promotion`.gold.dim_customer


In [0]:
%sql
SELECT
    customer_segment,
    COUNT(*) AS customers,
    ROUND(
        100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 2
    ) AS customer_pct
FROM `campaign&promotion`.gold.dim_customer
GROUP BY customer_segment
ORDER BY customers DESC;


### 6. Transaction profile

In [0]:
%sql
SELECT
    COUNT(*) AS total_transactions,
    COUNT(DISTINCT transaction_id) AS distinct_transactions,
    COUNT(DISTINCT customer_id) AS transacting_customers,
    SUM(CASE WHEN status = 'SUCCESS' THEN 1 ELSE 0 END) AS successful_transactions,
    SUM(CASE WHEN status <> 'SUCCESS' THEN 1 ELSE 0 END) AS unsuccessful_transactions,
    SUM(CASE WHEN status = 'SUCCESS' THEN amount ELSE 0 END) AS successful_transaction_value,
    SUM(CASE WHEN status = 'SUCCESS' THEN cashback ELSE 0 END) AS cashback
FROM `campaign&promotion`.gold.fct_transaction;
    

### 7. Participation profile

In [0]:
%sql
SELECT
    COUNT(*) AS participation_records,
    COUNT(DISTINCT campaign_id) AS campaigns,
    COUNT(DISTINCT customer_id) AS customers,
    SUM(CASE WHEN eligible_flag THEN 1 ELSE 0 END) AS eligible_records,
    SUM(CASE WHEN participated_flag THEN 1 ELSE 0 END) AS participant_records
FROM `campaign&promotion`.gold.fct_campaign_participation;

### 8. Redemption profile

In [0]:
%sql
SELECT
    COUNT(*) AS redemption_count,
    COUNT(DISTINCT campaign_id) AS campaigns,
    COUNT(DISTINCT customer_id) AS customers,
    SUM(reward_amount) AS total_reward_cost,
    SUM(cashback_amount) AS total_cashback,
    SUM(discount_amount) AS total_discount
FROM `campaign&promotion`.gold.fct_promotion_redemption;